In [0]:
%pip install catboost xgboost lightgbm --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [0]:
# Load train and test data from Unity Catalog
train_df = spark.table("science_home.`ml-dielectric`.train_data").toPandas()
test_df = spark.table("science_home.`ml-dielectric`.test_data").toPandas()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

Train shape: (5116, 94)
Test shape: (1280, 94)


,id,formula,band_gap,cbm,density_atomic,dielectric_constant,e_fermi,energy_above_hull,energy_per_atom,equilibrium_reaction_energy_per_atom,formation_energy_per_atom,g_reuss,g_voigt,g_vrh,homogeneous_poisson,k_reuss,k_voigt,k_vrh,nelements,nsites,uncorrected_energy_per_atom,universal_anisotropy,vbm,volume,bulk_modulus,density,poisson_ratio,shear_modulus,space_group,youngs_modulus,lattice_parameters,MagpieData_mean_Electronegativity,MagpieData_avg_dev_Electronegativity,MagpieData_minimum_Electronegativity,MagpieData_maximum_Electronegativity,MagpieData_range_Electronegativity,MagpieData_mean_AtomicWeight,MagpieData_avg_dev_AtomicWeight,MagpieData_minimum_AtomicWeight,MagpieData_maximum_AtomicWeight,...,MagpieData_maximum_GSbandgap,MagpieData_range_GSbandgap,MagpieData_mean_NdValence,MagpieData_avg_dev_NdValence,MagpieData_minimum_NdValence,MagpieData_maximum_NdValence,MagpieData_range_NdValence,MagpieData_mean_NsValence,MagpieData_avg_dev_NsValence,MagpieData_minimum_NsValence,MagpieData_maximum_NsValence,MagpieData_range_NsValence,MagpieData_mean_NpValence,MagpieData_avg_dev_NpValence,MagpieData_minimum_NpValence,MagpieData_maximum_NpValence,MagpieData_range_NpValence,MagpieData_mean_NUnfilled,MagpieData_avg_dev_NUnfilled,MagpieData_minimum_NUnfilled,MagpieData_maximum_NUnfilled,MagpieData_range_NUnfilled,MagpieData_mean_CovalentRadius,MagpieData_avg_dev_CovalentRadius,MagpieData_minimum_CovalentRadius,MagpieData_maximum_CovalentRadius,MagpieData_range_CovalentRadius,avg_s_valence_electrons,avg_p_valence_electrons,avg_d_valence_electrons,avg_f_valence_electrons,frac_s_valence_electrons,frac_p_valence_electrons,frac_d_valence_electrons,frac_f_valence_electrons,band_center,HOMO_energy,LUMO_energy,gap_AO,acsf_3761
0,mp-1070498,Rb2Te2Pt,-0.901793,-0.084780,1.921082,2.405881,0.518768,-0.186404,0.863588,0.050730,0.774359,-0.257589,0.015605,-0.735149,-0.257314,-0.127276,-0.012587,-0.805693,-0.240796,-0.916579,-2.627188,0.012505,0.605736,-0.518307,-0.805693,1.001670,0.501707,-0.906801,-0.321523,-0.916969,0.831808,-1.754445,-0.346487,-0.885403,-1.636482,-0.657672,2.589134,0.250622,2.919475,1.648767,...,-0.759858,-0.749343,1.374143,1.509929,-0.197424,0.838263,0.904796,-1.733833,1.172647,-0.687689,0.06631,0.696212,-1.032236,0.613275,-0.286435,-0.092688,0.112561,-0.592646,-0.806222,0.194837,-1.064062,-1.160555,2.274303,0.380113,2.096617,1.256225,-0.209878,-1.733833,-1.032236,1.374143,1.692288,-1.410977,-1.637261,1.185330,1.625460,-1.541595,0.852906,0.214086,-0.612131,-0.012564
1,mp-15567,Na3AuS2,-0.089327,-0.252607,0.469678,2.111637,-0.183831,-0.186404,0.900637,0.155368,0.680706,-0.257589,0.015605,-0.567415,-0.257314,-0.127276,-0.012587,-0.563946,-0.240796,-0.380266,-0.546952,0.012505,-0.159977,-0.083870,-0.563946,-0.098973,0.444052,-0.679154,0.943165,-0.660059,0.836354,-1.493947,0.250512,-0.657457,-1.092007,-0.394279,0.243715,1.148204,0.057195,1.683183,...,-0.061919,-0.039106,-0.227580,0.416798,-0.197424,0.838263,0.904796,-2.032192,0.999667,-0.687689,0.06631,0.696212,-1.338521,0.342004,-0.286435,-0.092688,0.112561,-0.901657,-0.858358,0.194837,-1.064062,-1.160555,1.174895,-0.514169,0.999962,-0.102180,-0.727339,-2.032192,-1.338521,-0.227580,1.342336,-0.798440,-1.242141,0.004316,2.559765,-1.322666,0.298803,1.129539,0.948208,-0.012564
2,mp-3592,KAuS5,-0.401510,-0.278313,0.782250,2.166812,0.035812,-0.186404,-1.307850,0.205709,1.358480,-0.257589,0.015605,-0.880841,-0.257314,-0.127276,-0.012587,-1.070050,-0.240796,0.845593,-0.591217,0.012505,0.053037,1.891178,-1.070050,-0.290862,0.099826,-1.073257,-0.308349,-1.119743,2.994288,-0.290233,-1.075335,-0.885403,-1.092007,-0.241788,0.298465,0.750482,0.472955,1.683183,...,-0.061919,-0.039106,-0.319845,0.223792,-0.197424,0.838263,0.904796,-0.327284,0.823157,-0.687689,0.06631,0.696212,0.411680,0.065196,-0.286435,-0.092688,0.112561,-0.460212,-0.911558,0.194837,-1.064062,-1.160555,0.537507,-0.452259,0.999962,0.828579,0.105097,-0.327284,0.411680,-0.319845,1.092371,-0.688107,-0.241798,-0.333117,1.699611,-0.21

In [0]:
# Define target column
TARGET = "dielectric_constant"

# Drop non-numeric columns
cols_to_drop = ["id", "formula"]
train_df = train_df.drop(columns=cols_to_drop, errors='ignore')
test_df = test_df.drop(columns=cols_to_drop, errors='ignore')

# Separate features and target
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

# Drop rows where target is missing
mask_train = y_train.notna()
mask_test = y_test.notna()
X_train, y_train = X_train[mask_train], y_train[mask_train]
X_test, y_test = X_test[mask_test], y_test[mask_test]

print(f"Features: {X_train.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Features: 91
Training samples: 5116
Testing samples: 1280


In [0]:
# NOTE: Target (dielectric_constant) is LOG-TRANSFORMED.
# Use np.exp(predictions) to get actual dielectric constant values.

# Drop non-numeric columns (XGBoost/GradientBoosting require numeric input)
X_train = X_train.select_dtypes(include=[np.number])
X_test = X_test[X_train.columns]

mlflow.set_experiment("/Users/j.krishna@exomatter.ai/dielectric_model_comparison")

# Define two candidate models
models = {
    "XGBoost": XGBRegressor(
        random_state=42, n_estimators=1000, max_depth=5, learning_rate=0.05,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1, reg_lambda=10, objective="reg:squarederror", n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        random_state=42, n_estimators=1000, max_depth=5, learning_rate=0.05,
        subsample=0.8, max_features=0.8, min_samples_leaf=5
    ),
}

# Train, evaluate, and log each model
results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Log-space metrics
        log_test_r2 = r2_score(y_test, y_test_pred)
        log_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

        # Original-scale metrics
        orig_test_r2 = r2_score(np.exp(y_test), np.exp(y_test_pred))
        orig_test_rmse = np.sqrt(mean_squared_error(np.exp(y_test), np.exp(y_test_pred)))
        orig_test_mae = mean_absolute_error(np.exp(y_test), np.exp(y_test_pred))

        # Log to MLflow
        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            "log_test_r2": log_test_r2, "log_test_rmse": log_test_rmse,
            "orig_test_r2": orig_test_r2, "orig_test_rmse": orig_test_rmse,
            "orig_test_mae": orig_test_mae,
        })
        mlflow.set_tag("model_type", name)
        mlflow.set_tag("target_transform", "log")

        signature = mlflow.models.infer_signature(X_train, y_test_pred)
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)

        results[name] = {"model": model, "log_test_r2": log_test_r2, "orig_test_r2": orig_test_r2, "orig_test_rmse": orig_test_rmse}

# Compare and select best
print(f"{'='*65}")
print(f" {'Model':<20} {'Test R\u00b2 (log)':>13} {'Test R\u00b2 (orig)':>14} {'Test RMSE (orig)':>16}")
print(f"{'='*65}")
for name, m in results.items():
    print(f" {name:<20} {m['log_test_r2']:>13.4f} {m['orig_test_r2']:>14.4f} {m['orig_test_rmse']:>16.4f}")

best_name = max(results, key=lambda k: results[k]["log_test_r2"])
best_model = results[best_name]["model"]
print(f"\n>>> Best model: {best_name} (Test R\u00b2 = {results[best_name]['log_test_r2']:.4f})")

2026/06/16 11:24:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-53810b9a-9c4d.cloud.databricks.com/ml/experiments/3103778536368019/models/m-9463d5b0b760465ab85288241be27a87?o=7474645731664966
2026/06/16 11:26:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-53810b9a-9c4d.cloud.databricks.com/ml/experiments/3103778536368019/models/m-ad34c6e392b44f39bf29dc1f305d770f?o=7474645731664966


 Model                Test R² (log) Test R² (orig) Test RMSE (orig)
 XGBoost                     0.7567         0.6701           3.5247
 GradientBoosting            0.7513         0.6578           3.5900

>>> Best model: XGBoost (Test R² = 0.7567)


In [0]:
from sklearn.model_selection import GridSearchCV

# GridSearchCV for XGBoost
# Reduced grid to keep runtime practical (~30-45 min on serverless)
param_grid = {
    "n_estimators": [1000, 2000],
    "max_depth": [3, 5],
    "learning_rate": [0.01, 0.05],
    "subsample": [0.7, 0.8],
    "colsample_bytree": [0.7, 0.8],
    "reg_alpha": [1],
    "reg_lambda": [5],
}

n_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Running GridSearchCV on: XGBoost")
print(f"Parameter grid: {n_combinations} combinations x 5 folds = {n_combinations * 5} fits")

grid_search = GridSearchCV(
    XGBRegressor(random_state=42, objective="reg:squarederror", n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2,
    refit=True,
)
grid_search.fit(X_train, y_train)

print(f"\n{'='*50}")
print(f" GridSearchCV Results (XGBoost)")
print(f"{'='*50}")
print(f" Best CV R\u00b2: {grid_search.best_score_:.4f}")
print(f" Best Parameters:")
for k, v in grid_search.best_params_.items():
    print(f"   {k}: {v}")

Running GridSearchCV on: XGBoost
Parameter grid: 32 combinations x 5 folds = 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits

 GridSearchCV Results (XGBoost)
 Best CV R²: 0.7599
 Best Parameters:
   colsample_bytree: 0.8
   learning_rate: 0.05
   max_depth: 5
   n_estimators: 2000
   reg_alpha: 1
   reg_lambda: 5
   subsample: 0.8


In [0]:
# Evaluate best tuned model
tuned_model = grid_search.best_estimator_

y_train_pred = tuned_model.predict(X_train)
y_test_pred = tuned_model.predict(X_test)

# Metrics in both spaces
metrics = {
    "log_train_r2": r2_score(y_train, y_train_pred),
    "log_test_r2": r2_score(y_test, y_test_pred),
    "log_test_rmse": np.sqrt(mean_squared_error(y_test, y_test_pred)),
    "orig_train_r2": r2_score(np.exp(y_train), np.exp(y_train_pred)),
    "orig_test_r2": r2_score(np.exp(y_test), np.exp(y_test_pred)),
    "orig_test_rmse": np.sqrt(mean_squared_error(np.exp(y_test), np.exp(y_test_pred))),
    "orig_test_mae": mean_absolute_error(np.exp(y_test), np.exp(y_test_pred)),
    "cv_r2": grid_search.best_score_,
}

# Log final model to MLflow
with mlflow.start_run(run_name=f"{best_name}_tuned_final"):
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metrics(metrics)
    mlflow.set_tag("model_type", f"{best_name}_tuned")
    mlflow.set_tag("target_transform", "log")
    mlflow.set_tag("tuning_method", "GridSearchCV")

    signature = mlflow.models.infer_signature(X_train, y_test_pred)
    mlflow.sklearn.log_model(tuned_model, artifact_path="model", signature=signature)

# Print summary
print(f"{'='*60}")
print(f" Final Model: {best_name} (tuned via GridSearchCV)")
print(f"{'='*60}")
print(f" Log-space:")
print(f"   Train R\u00b2: {metrics['log_train_r2']:.4f} | Test R\u00b2: {metrics['log_test_r2']:.4f}")
print(f"   Test RMSE: {metrics['log_test_rmse']:.4f}")
print(f" Original-scale (np.exp applied):")
print(f"   Train R\u00b2: {metrics['orig_train_r2']:.4f} | Test R\u00b2: {metrics['orig_test_r2']:.4f}")
print(f"   Test RMSE: {metrics['orig_test_rmse']:.4f} | Test MAE: {metrics['orig_test_mae']:.4f}")
print(f" CV R\u00b2: {metrics['cv_r2']:.4f}")

2026/06/16 11:54:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-53810b9a-9c4d.cloud.databricks.com/ml/experiments/3103778536368019/models/m-6f68f8c3d158494486d2f262397f89ac?o=7474645731664966


 Final Model: XGBoost (tuned via GridSearchCV)
 Log-space:
   Train R²: 0.9896 | Test R²: 0.7604
   Test RMSE: 0.2560
 Original-scale (np.exp applied):
   Train R²: 0.9823 | Test R²: 0.6759
   Test RMSE: 3.4936 | Test MAE: 2.1544
 CV R²: 0.7599


In [0]:
# Register the tuned XGBoost model to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "science_home.ml-dielectric.xgboost_dielectric"

# Get the run ID from the last logged run
run_id = mlflow.search_runs(
    experiment_names=["/Users/j.krishna@exomatter.ai/dielectric_model_comparison"],
    filter_string=f"tags.model_type = '{best_name}_tuned'",
    order_by=["start_time DESC"],
    max_results=1,
).iloc[0].run_id

model_uri = f"runs:/{run_id}/model"

# Register the model
model_version = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(f"{'='*60}")
print(f" Model Registered to Unity Catalog")
print(f"{'='*60}")
print(f" Name:    {MODEL_NAME}")
print(f" Version: {model_version.version}")
print(f" URI:     {model_uri}")
print(f"\n Note: Target is log-transformed. Apply np.exp() to predictions.")

Registered model 'science_home.ml-dielectric.xgboost_dielectric' already exists. Creating a new version of this model...
2026/06/16 11:54:35 WARNING mlflow.tracking._model_registry.fluent: Run with id b054db32eaa3410b89d3ece1d32039e4 has no artifacts at artifact path 'model', registering model based on models:/m-6f68f8c3d158494486d2f262397f89ac instead


Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

🔗 Created version '4' of model 'science_home.ml-dielectric.xgboost_dielectric': https://dbc-53810b9a-9c4d.cloud.databricks.com/explore/data/models/science_home/ml-dielectric/xgboost_dielectric/version/4?o=7474645731664966


 Model Registered to Unity Catalog
 Name:    science_home.ml-dielectric.xgboost_dielectric
 Version: 4
 URI:     runs:/b054db32eaa3410b89d3ece1d32039e4/model

 Note: Target is log-transformed. Apply np.exp() to predictions.
